# DocVQA Zero-Shot Baseline - Google Colab (Production Ready)

Complete setup and evaluation pipeline for the VisionDocPhi-3.5 project.

This notebook:
- ✅ Sets up the production-ready project structure
- ✅ Installs all dependencies
- ✅ Loads Phi-3.5 Vision model
- ✅ Runs zero-shot VQA on DocVQA dataset
- ✅ Calculates metrics (ANLS, Exact Match)
- ✅ Saves results

## Step 1: Clone from GitHub & Setup Project

In [ ]:
import os
import sys

# ============ STEP 1: Clone GitHub Repository ============
PROJECT_NAME = "VisionDocPhi-3.5"
GITHUB_REPO = "https://github.com/mokshu7k/VisionDocPhi-3.5.git"
PROJECT_PATH = f"/content/{PROJECT_NAME}"

# Clone repository
if not os.path.exists(PROJECT_PATH):
    print("📥 Cloning repository from GitHub...")
    !git clone {GITHUB_REPO} {PROJECT_PATH}
    print("✅ Repository cloned successfully!\n")
else:
    print(f"✓ Repository already exists at {PROJECT_PATH}\n")

# Change to project directory
os.chdir(PROJECT_PATH)
sys.path.insert(0, PROJECT_PATH)

print(f"📂 Working directory: {os.getcwd()}\n")

## Step 2: Pull Latest Changes (Optional)

In [ ]:
# Pull latest changes from GitHub (run this if you made updates to the repo)
# !git pull origin main
print("✓ Ready to pull updates if needed: git pull origin main")

## Step 3: Install Dependencies

In [ ]:
# Install dependencies from requirements.txt
print("📦 Installing dependencies...\n")
!pip install -q -r requirements.txt

# Optional: Install flash-attn for faster GPU inference (comment out if it fails)
print("\n⚡ Installing FlashAttention2 for GPU optimization...")
!pip install -q flash-attn --no-build-isolation 2>/dev/null || echo "⚠️  FlashAttention2 not available (will use eager attention)"

print("\n✅ All dependencies installed!")

✅ Dependencies installed!


## Step 4: Verify Project Setup

In [3]:
import os
import sys
from pathlib import Path
import torch

print("\n" + "="*70)
print("🔍 VERIFICATION CHECKLIST")
print("="*70 + "\n")

# Check directory structure
print("📁 Checking project structure...")
required_dirs = ['config', 'src', 'scripts', 'notebooks', 'data/raw', 'data/outputs']

for dir_name in required_dirs:
    if os.path.exists(dir_name):
        print(f"  ✓ {dir_name}/")
    else:
        print(f"  ✗ {dir_name}/ NOT FOUND")

print("\n📦 Checking PyTorch...")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"  Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

print("\n📊 Checking dataset files...")
if os.path.exists('data/raw/spdocvqa_qas/val_v1.0_withQT.json'):
    print("  ✓ Validation annotations found")
else:
    print("  ✗ Validation annotations NOT found")
    print("    Download from: https://rrc.cvc.uab.es/?ch=17")

if os.path.exists('data/raw/spdocvqa_images'):
    num_images = len(os.listdir('data/raw/spdocvqa_images'))
    print(f"  ✓ Images directory ({num_images} items)")
else:
    print("  ✗ Images directory NOT found")

print("\n" + "="*70)


🔍 VERIFICATION CHECKLIST

📁 Checking project structure...
  ✗ config/ NOT FOUND
  ✗ src/ NOT FOUND
  ✗ scripts/ NOT FOUND
  ✗ notebooks/ NOT FOUND
  ✗ data/raw/ NOT FOUND
  ✗ data/outputs/ NOT FOUND

📦 Checking PyTorch...
  PyTorch: 2.12.0+cpu
  CUDA available: False
  Device: cpu

📊 Checking dataset files...
  ✗ Validation annotations NOT found
    Download from: https://rrc.cvc.uab.es/?ch=17
  ✗ Images directory NOT found



## Step 5: Import Project Modules

In [ ]:
print("Importing project modules...\n")

Importing project modules...



ModuleNotFoundError: No module named 'config'

## Step 6: Display Configuration

In [5]:
print("\n" + "="*70)
print("⚙️  PROJECT CONFIGURATION")
print("="*70)

print(f"\n📍 Model: {MODEL_NAME}")
print(f"💾 Device: {DEVICE or ('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"📂 Project Root: {PROJECT_ROOT}")
print(f"🖼️  Images: {IMAGES_DIR}")
print(f"📋 Annotations: {VAL_ANNOTATIONS}")
print(f"⚡ Batch Size: {BATCH_SIZE}")

print("\n" + "="*70)


⚙️  PROJECT CONFIGURATION


NameError: name 'MODEL_NAME' is not defined

## Step 7: Get Dataset Statistics

In [ ]:
print("📊 Dataset Statistics:\n")

stats = get_dataset_stats(str(VAL_ANNOTATIONS))

print(f"  Total Samples: {stats['total_samples']}")
print(f"  Question Types: {stats['num_question_types']}")

if stats['question_types']:
    print(f"\n  Breakdown by Type:")
    for qtype, count in sorted(stats['question_types'].items(), key=lambda x: x[1], reverse=True):
        pct = (count / stats['total_samples']) * 100
        print(f"    - {qtype}: {count} ({pct:.1f}%)")

print()

## Step 8: Initialize Model

In [ ]:
# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

print("🚀 Initializing Phi-3.5 Vision model...\n")
print(f"Device: {device}")
print(f"Model: {MODEL_NAME}\n")

inference = DocVQAInference(model_name=MODEL_NAME, device=device)

## Step 9: Quick Test (5 Samples)

In [ ]:
import json
from PIL import Image
from tqdm import tqdm

print("\n" + "="*70)
print("🧪 QUICK TEST (First 5 Samples)")
print("="*70 + "\n")

# Create dataloader
dataloader = create_dataloader(
    annotations_file=str(VAL_ANNOTATIONS),
    image_dir=str(IMAGES_DIR),
    split='val',
    batch_size=1,
    num_workers=0,
    shuffle=False
)

test_results = []

for i, batch in enumerate(dataloader):
    if i >= 5:  # Only test 5 samples
        break
    
    for sample in batch:
        image = sample['image']
        question = sample['question']
        ground_truth = sample['answers'][0] if sample['answers'] else "N/A"
        question_id = sample['question_id']
        
        # Generate answer
        predicted_answer = inference.generate_answer(image, question)
        
        print(f"Sample {i+1}:")
        print(f"  Q: {question}")
        print(f"  Predicted: {predicted_answer}")
        print(f"  Ground Truth: {ground_truth}")
        print()
        
        test_results.append({
            'question': question,
            'predicted': predicted_answer,
            'ground_truth': ground_truth
        })

print("✅ Quick test completed!")

## Step 10: Full Evaluation Pipeline

**Note:** This runs the complete evaluation. Depending on dataset size, it may take significant time.

In [ ]:
print("\n" + "="*70)
print("📊 FULL EVALUATION PIPELINE")
print("="*70 + "\n")

# Run using the production pipeline
eval_results = run_zero_shot_baseline(
    split="val",
    num_samples=None,  # Use all samples
    save_results=True
)

print("\n✅ Evaluation completed!")

## Step 11: Display Results

In [ ]:
print("\n" + "="*70)
print("📈 FINAL RESULTS")
print("="*70 + "\n")

# Extract metrics
metrics = {k: v for k, v in eval_results.items() if k != 'results'}

print("Metrics:")
for metric_name, metric_value in metrics.items():
    if isinstance(metric_value, float):
        print(f"  {metric_name}: {metric_value:.4f}")
    else:
        print(f"  {metric_name}: {metric_value}")

print(f"\nResults saved to: data/outputs/")
print(f"  - predictions_zeroshot.json (detailed predictions)")
print(f"  - results_zeroshot.json (metrics summary)")

print("\n" + "="*70)

## Step 12: Download Results (Optional)

In [ ]:
from google.colab import files
import os

print("\n📥 Downloading results...\n")

# Download results if they exist
results_dir = 'data/outputs'
if os.path.exists(results_dir):
    for file in os.listdir(results_dir):
        if file.endswith('.json'):
            file_path = os.path.join(results_dir, file)
            files.download(file_path)
            print(f"✓ {file}")
    print("\n✅ Download complete!")
else:
    print("⚠️  No results directory found. Run evaluation first!")

## Troubleshooting

### Common Issues:

1. **"FileNotFoundError" for data files**
   - Download dataset: https://rrc.cvc.uab.es/?ch=17
   - Extract to: `data/raw/spdocvqa_images/` and `data/raw/spdocvqa_qas/`

2. **"ModuleNotFoundError" when importing**
   - Ensure you're in the correct project directory
   - Check that all `__init__.py` files exist in src/ subdirectories

3. **Out of Memory (OOM) Error**
   - Runtime → Change runtime type → Select GPU T4
   - Reduce number of samples: `num_samples=100`

4. **Model Download Fails**
   - Check internet connection
   - Try again (HuggingFace can be slow)

### To Use Different Dataset Split:

```python
# Run on test set
eval_results = run_zero_shot_baseline(split="test")
```

### To Evaluate on Subset:

```python
# Evaluate only first 100 samples
eval_results = run_zero_shot_baseline(split="val", num_samples=100)
```